# Robô de triagem da Cobratec — modelo rodando no Colab

Sobe um **Ollama com GPU** aqui no Colab e o entrega ao inventário por um túnel
autenticado. Serve para a máquina do escritório não ter que segurar o modelo.

```
WhatsApp → WAHA → inventário (/chat) → túnel → [Colab] proxy → Ollama (GPU)
```

## Leia antes de rodar

**Isto é o caminho de TESTE.** A decisão 31 escolheu rodar o modelo dentro da
empresa porque a fala do devedor não deve sair dela. Apontar o inventário para
cá manda a mensagem do devedor para uma VM do Google — o que é aceitável para
**medir modelo e afinar a triagem com mensagens inventadas**, e não é aceitável
com devedor real. A tela `/chat → Conexão` avisa em vermelho quando o modelo
está fora da rede; se o aviso estiver aparecendo em produção, algo está errado.

O que o robô faz continua o mesmo, aqui ou lá: ele **não fala de valor, acordo
nem pagamento** — isso é barrado por código no inventário (`lib/chat-bot.ts`),
não pelo modelo. Trocar de máquina não afrouxa nada disso.

**Limites do Colab, que você vai encontrar:** a sessão cai sozinha depois de
~90 min sem uso e tem teto de ~12h; o endereço do túnel **muda a cada vez** que
você roda de novo (e o `.env` do inventário precisa ser atualizado); e servir
tráfego contínuo não é o uso que o Colab se propõe a suportar. Para valer,
o modelo mora numa máquina sua.

## Como usar

1. **Ambiente de execução → Alterar o tipo → GPU (T4)**. Sem GPU não vale a
   pena: a CPU do Colab é mais lenta que a do escritório.
2. Rode as células **1 a 4** em ordem.
3. Copie o bloco que a célula 4 imprime para o `.env` do inventário e recrie o
   app (`docker compose up -d`).
4. Deixe a célula 5 rodando e **a aba aberta** — é o que segura a sessão viva.

## 1. GPU e instalação do Ollama

In [ ]:
import subprocess, sys

# Sem GPU o Colab não ajuda em nada: a CPU dele é mais fraca que a de um
# desktop de escritório. Melhor descobrir agora que depois de baixar 2GB.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("SEM GPU. Ambiente de execução → Alterar o tipo de ambiente → T4 GPU.")
    print("Rodar em CPU aqui é mais lento que rodar na máquina do escritório.")
    sys.exit(1)
print("GPU:", gpu.stdout.strip())

!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

## 2. Subir o Ollama e baixar o modelo

`llama3.2:3b` é o padrão aqui **porque tem GPU**. Na máquina do escritório o
padrão é o `1b`, que cabe em CPU. É justamente essa diferença que este notebook
existe para você medir: se o 3B com GPU não triar visivelmente melhor que o 1B
local, não vale a dependência de um túnel.

In [ ]:
import os, time, subprocess, requests

MODELO = "llama3.2:3b"  # troque para medir: llama3.2:1b, qwen2.5:3b, gemma2:2b

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# Mantém o modelo na GPU entre mensagens. Sem isto, cada pausa na conversa paga
# a carga de novo — o mesmo motivo do keep_alive em lib/chat-bot.ts.
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"

servidor = subprocess.Popen(["ollama", "serve"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("o Ollama não subiu")

print("Ollama de pé. Baixando", MODELO, "— alguns minutos na primeira vez.")
!ollama pull {MODELO} 2>&1 | tail -2

## 3. Medir antes de confiar

Roda o modelo com as falas que mais aparecem e mostra **quanto tempo cada uma
leva**. O inventário desiste em **45s** e manda para a fila, então qualquer
resposta acima disso é uma conversa que a operadora vai atender de qualquer
jeito — só que depois de a pessoa esperar.

Repare que as falas graves ("quanto eu devo") **nem chegariam ao modelo** em
produção: o inventário as manda para a fila antes, sem inferência. Estão aqui
só para você ver o que o modelo faria se dependesse dele — e é por isso que não
depende.

In [ ]:
import json, time, requests

FALAS = [
    "oi",
    "bom dia, tudo bem?",
    "quem fala?",
    "o que é a cobratec?",
    "quanto eu devo?",             # em produção: fila, sem passar pelo modelo
    "já paguei isso mês passado",  # em produção: fila, sem passar pelo modelo
]

# Cópia enxuta do prompt de lib/chat-bot.ts, só para a medição ser honesta:
# prompt curto é parte do desenho, e medir com outro texto mediria outra coisa.
SISTEMA = (
    "Você é a recepcionista virtual da Cobratec no WhatsApp. Fale com a pessoa, "
    "com educação.\n\nVocê não tem acesso a dado nenhum: nem cadastro, nem valor, "
    "nem prazo. Nunca invente.\n\nEscale para uma atendente humana sempre que o "
    "assunto for a dívida ou a pessoa estiver irritada.\n\nResponda SEMPRE em JSON: "
    '{"resposta":"o que dizer","escalar":true ou false,"motivo":"por que escalou"}'
)

print(f"{'fala':<32} {'seg':>6}  resposta")
print("-" * 92)
piores = []
for fala in FALAS:
    t0 = time.time()
    r = requests.post("http://127.0.0.1:11434/api/chat", timeout=180, json={
        "model": MODELO,
        "messages": [{"role": "system", "content": SISTEMA},
                     {"role": "user", "content": fala}],
        "stream": False, "format": "json", "keep_alive": "60m",
        "options": {"temperature": 0, "num_predict": 120},
    })
    seg = time.time() - t0
    piores.append(seg)
    try:
        d = json.loads(r.json()["message"]["content"])
        saida = f"{'ESCALA' if d.get('escalar') else 'responde'}: {d.get('resposta', '')[:44]}"
    except Exception:
        saida = f"FORA DO FORMATO (o inventário escalaria): {r.text[:40]}"
    print(f"{fala:<32} {seg:>6.1f}  {saida}")

pior = max(piores)
print("-" * 92)
print(f"pior caso: {pior:.1f}s  (teto do inventário: 45s)")
print("OK: cabe com folga." if pior < 20 else
      ("Apertado: passe para um modelo menor." if pior < 45 else
       "NÃO SERVE: acima do teto, tudo cairia na fila."))

## 4. Abrir a porta para o inventário

O Ollama **não tem autenticação nenhuma**. Num túnel público isso seria um
modelo aberto para quem achasse o endereço, e o endereço não é secreto.

Então quem atende o túnel não é o Ollama: é um proxy de vinte linhas que exige
um `Bearer` e só deixa passar **duas rotas** — conversar e listar modelos. Sem
isso, um `DELETE /api/delete` da internet apagaria o modelo no meio do
atendimento.

In [ ]:
import hmac, re, secrets, subprocess, threading, time, requests
from flask import Flask, request, Response

TOKEN = secrets.token_urlsafe(32)
OLLAMA = "http://127.0.0.1:11434"

# Só o que o inventário usa. Lista de permissão, não de bloqueio: rota nova do
# Ollama nasce fechada em vez de nascer exposta.
LIBERADAS = {("POST", "/api/chat"), ("GET", "/api/tags")}

app = Flask(__name__)

@app.route("/<path:caminho>", methods=["GET", "POST"])
def repassar(caminho):
    rota = "/" + caminho
    if (request.method, rota) not in LIBERADAS:
        return Response('{"error":"rota fechada"}', 403, mimetype="application/json")

    # compare_digest: comparação de segredo não vaza o tamanho do acerto.
    enviado = request.headers.get("Authorization", "")
    if not hmac.compare_digest(enviado, f"Bearer {TOKEN}"):
        return Response('{"error":"token inválido"}', 401, mimetype="application/json")

    resp = requests.request(request.method, OLLAMA + rota,
                            data=request.get_data(), timeout=180,
                            headers={"content-type": "application/json"})
    return Response(resp.content, resp.status_code, mimetype="application/json")

threading.Thread(
    target=lambda: app.run(host="127.0.0.1", port=8000, threaded=True),
    daemon=True,
).start()
time.sleep(2)

# Túnel do Cloudflare: endereço público sem conta e sem cadastro. Ele MUDA a
# cada execução — é a fricção principal deste caminho.
!wget -q -O /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cf.deb > /dev/null 2>&1

tunel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url = None
for linha in tunel.stdout:
    achou = re.search(r"https://[-\w]+\.trycloudflare\.com", linha)
    if achou:
        url = achou.group(0)
        break
if not url:
    raise RuntimeError("o túnel não subiu — rode a célula de novo")

conferencia = requests.get(f"{url}/api/tags", timeout=30,
                           headers={"Authorization": f"Bearer {TOKEN}"})
print("túnel respondendo:", conferencia.status_code == 200)
print("porta fechada sem token:", requests.get(f"{url}/api/tags", timeout=30).status_code == 401)

print("\n" + "=" * 74)
print("Cole no .env do inventário e rode:  docker compose up -d")
print("=" * 74)
print(f'OLLAMA_URL="{url}"')
print(f'OLLAMA_MODELO="{MODELO}"')
print(f'OLLAMA_TOKEN="{TOKEN}"')
print("=" * 74)
print("Este endereço morre quando a sessão do Colab cair. Quando isso acontecer,")
print("rode o notebook de novo e troque as três linhas — ou apague OLLAMA_URL,")
print("que o atendimento volta a cair na fila da operadora sem quebrar nada.")

## 5. Segurar a sessão viva

Deixe esta célula rodando **e a aba aberta**. Ela também é o seu monitor: mostra
quantas mensagens o inventário mandou e avisa quando o modelo cair.

Para desligar tudo: interrompa a célula e feche a aba. No inventário, apague
`OLLAMA_URL` do `.env` — o atendimento volta inteiro para a fila da operadora,
sem quebrar nada.

In [ ]:
import time, requests
from datetime import datetime, timedelta, timezone

BRASIL = timezone(timedelta(hours=-3))
inicio = time.time()

while True:
    try:
        vivo = requests.get(f"{OLLAMA}/api/tags", timeout=5).status_code == 200
    except Exception:
        vivo = False

    horas = (time.time() - inicio) / 3600
    agora = datetime.now(BRASIL).strftime("%H:%M")
    estado = "ok" if vivo else "MODELO FORA DO AR — o inventário está escalando tudo"
    print(f"\r{agora}  de pé há {horas:4.1f}h  {estado}   ", end="")

    if horas > 11.5:
        print("\nperto do teto de 12h do Colab: a sessão vai cair em breve.")
    time.sleep(60)